# Red Shift Training Notebook

This notebook has two tracks. Track A is the implemented GPU policy trainer used for committed plots. Track B documents the LLM GRPO/Unsloth path to wire once model rollouts are ready.

## Setup

On Colab, uncomment the install cell. Locally or on the server, use the repo environment.

In [ ]:
# Colab install, if needed:
# !pip install -q openenv-core fastapi uvicorn pydantic requests pyyaml matplotlib nbformat torch
# !pip install -q git+https://github.com/huggingface/trl.git unsloth trackio

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print(ROOT)

## Track A: Runnable GPU Policy Training

This trains a neural defender over Red Shift tool/service choices and writes checkpoint, summary JSON, and PNG plots.

In [ ]:
!PYTHONPATH=src python scripts/train_redshift_policy.py --steps 800 --batch-size 96 --lr 0.003

In [ ]:
from pathlib import Path
import json
summary = json.loads(Path("training_results/gpu_policy/summary.json").read_text())
print("device:", summary["device"], summary["cuda_device_name"])
print("baseline:", summary["baseline"]["mean"])
print("trained:", summary["trained"]["mean"])

In [ ]:
from IPython.display import Image, display
display(Image("docs/plots/gpu_policy_training_curve.png"))
display(Image("docs/plots/gpu_policy_baseline_vs_trained.png"))

## Track B: LLM GRPO/Unsloth Wiring Notes

The intended next LLM trainer maps model text to `Action(command=...)`, groups K rollouts per prompt, and uses the OpenEnv reward as the GRPO reward function. Stability settings from the spec: variance filtering, asymmetric clipping, and KL against the reference model. The environment is already fast enough for this because simulation is pure Python.

In [ ]:
# Sketch only: keep disabled until model rollout parser is connected.
# from trl import GRPOTrainer, GRPOConfig
# from unsloth import FastLanguageModel
#
# def reward_func(prompts, completions, **kwargs):
#     rewards = []
#     for task_id, completion in zip(kwargs["task_id"], completions):
#         env = OnCallRedShiftEnv()
#         env.reset(task_id=task_id)
#         for command in parse_commands(completion):
#             obs = env.step(Action(command=command))
#         rewards.append(float(obs.reward or 0.0))
#     return rewards
#
# trainer = GRPOTrainer(model=model, args=GRPOConfig(...), reward_funcs=reward_func, train_dataset=dataset)
# trainer.train()